# Session 2 — Talking to the machines

**Week 1 · Session 2 · Big idea: messages, roles & system prompts**
**Project version: v0.1 — the scripted support replier**

Last night you watched a model do seven impressive things. Tonight you make it do *one* thing
reliably, in ShopWise's voice, without promising customers money it cannot give them.

**How to read this** — run the cells top to bottom. Outputs are committed, so this reads without a
key. Model replies vary between runs; by section 7 you will know exactly why. 💳 marks a cell needing
a billed key — those explain themselves instead of crashing.

**Today's words:** message, role, system prompt, token, stateless, context window. No frameworks —
we write the raw SDK so that session 4 can show you what a framework takes away.

## 1. Setup

The key lives in `.env`, one folder up. Never in the notebook — a pasted key gets committed,
pushed, scraped and revoked, in that order.

> **Use `GEMINI_API_KEY`.** The SDK also accepts `GOOGLE_API_KEY` and **prefers it when both are
> set** — so a leftover key from another tutorial spends someone else's quota instead of failing
> cleanly. The cell below warns you if you have one.

In [2]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from google import genai

ENV_PATH = Path.cwd().parent / ".env"     # session-02/ -> .env
load_dotenv(ENV_PATH)

print(f"key found: {bool(os.getenv('GEMINI_API_KEY'))}")
if os.getenv("GOOGLE_API_KEY"):
    print("WARNING: GOOGLE_API_KEY is also set and takes precedence. Unset it.")

client = genai.Client()
print("client ready")

key found: True
client ready


### Model IDs live in one place

Google retires models. With the ID scattered through forty cells that is a 2am migration; with two
constants it is one edit. From section 9 these same names live in `project/config.py`, read from the
environment, and they stay there for the rest of the course.

In [4]:
MODEL        = "gemini-3.1-flash-lite"   # course default: fast, cheap, free tier
MODEL_STRONG = "gemini-3.7-flash"        # when the default is not good enough


def explain_error(err: Exception) -> str:
    '''Turn an API exception into one actionable sentence.'''
    text = str(err)
    if "RESOURCE_EXHAUSTED" in text or "429" in text:
        return "Rate limit, not a bug. Wait a minute and re-run."
    if "403" in text or "PERMISSION_DENIED" in text or "billed" in text.lower():
        return ("Needs a billing-enabled key. The committed output above shows what it "
                "returns — nothing in this course requires you to enable billing.")
    if "404" in text or "NOT_FOUND" in text:
        return "That model ID no longer exists — which is why IDs live in one cell."
    if "API key" in text:
        return "No key found. Check .env and re-run the setup cell."
    return f"Unexpected — read it, it usually says what is wrong:\n{text[:300]}"


print(f"default: {MODEL}  |  stronger: {MODEL_STRONG}")

default: gemini-3.1-flash-lite  |  stronger: gemini-3.7-flash


## 2. The promise from last night

Session 1 ended on a specific promise:

> *"Tomorrow we ask it about ShopWise's actual return policy, and it will answer — fluently, warmly,
> with total confidence, and completely invented."*

Same question, twice. The only difference is one sentence of instruction.

In [3]:
QUESTION = "What is ShopWise's return policy?"

bare = client.models.generate_content(model=MODEL, contents=QUESTION)
print(bare.text)

To provide you with the correct return policy, I need to clarify which **"ShopWise"** you are referring to, as there are a few different businesses using that name globally:

### 1. If you mean Shopwise (The supermarket chain in the Philippines)
Shopwise generally adheres to the following guidelines for returns and exchanges:
*   **Timeframe:** Items can typically be returned or exchanged within **7 days** of purchase.
*   **Condition:** Items must be in their original packaging, unused, and with all tags/labels intact.
*   **Requirements:** You **must** present your original official receipt.
*   **Exclusions:** Perishable goods (like fresh produce, meat, or dairy), sale items, and personal care items (like underwear) are often non-returnable unless they are defective or expired at the time of purchase.
*   **Process:** You should bring the item and receipt to the Customer Service counter of the branch where the item was purchased.

### 2. If you mean an online store or app named "Sho

Not what most people predict: it **hedged**, and asked which ShopWise you mean. It may even have
quoted the genuine policy of a real supermarket chain called Shopwise in the Philippines.

> **Keep this in mind all evening:** *a true fact about the wrong company is indistinguishable from
> an invented fact about the right one.*

Now watch what one sentence does.

In [4]:
ROLE = "You are ShopWise customer support, replying to a customer."

framed = client.models.generate_content(
    model=MODEL, contents=QUESTION,
    config={"system_instruction": ROLE},   # the ONLY difference
)
print(framed.text)

Hello! Thank you for reaching out to ShopWise. I’d be happy to clarify our return policy for you.

At ShopWise, we want you to be completely satisfied with your purchase. Here are the key details of our return policy:

*   **Return Window:** You can return most items within **30 days** of the delivery date.
*   **Condition:** Items must be in their original, unused condition, including all original packaging, tags, and accessories.
*   **Proof of Purchase:** Please have your order number or a copy of your receipt ready to initiate the return.
*   **Refunds:** Once we receive and inspect your returned item, we will process your refund to the original payment method. Please note that it typically takes 5–7 business days for the funds to appear in your account, depending on your bank.
*   **Exclusions:** Certain items, such as intimate apparel, clearance/final sale items, or personalized products, may not be eligible for return. These will be clearly marked on the product page.

**How to 

### That is the whole session, in two cells

Confident, warm, well-formatted — and **invented**. A 30-day window, eligibility conditions, a
returns portal, a refund timeline. ShopWise has no such policy; it is a fictional store whose only
facts live in `data/`.

The model's knowledge did not change between those two cells. What changed is *which continuation is
plausible*. A support agent does not hedge about their own return policy — so once you ask it to be
one, it states a policy.

**The instruction that makes an assistant useful is the same one that removes its reluctance to
invent.** Hallucination is structural — a property of the application you build, not a prompt you can
fix. Section 8 collects everything a prompt cannot fix, each mapped to the session that fixes it.

## 3. Recall and decide

Last night covered the landscape — providers, open weights, leaderboards, free tiers. **We are not
repeating it.** One decision instead.

Choosing a model = **capability × price × speed × context size × provider trust**. The honest rule:
try the cheap model first, move up only when you can name a task it fails.

In [5]:
import yaml

TICKETS = yaml.safe_load((Path.cwd().parent / "data" / "tickets.yml").read_text(encoding="utf-8"))
by_id = {t["id"]: t for t in TICKETS}

tk = by_id["TCK-003"]           # the furious all-caps one
prompt = f"Summarise this support ticket in one sentence, then rate its urgency 1-5.\n\n{tk['body']}"

for tier in (MODEL, MODEL_STRONG):
    reply = client.models.generate_content(model=tier, contents=prompt)
    u = reply.usage_metadata
    print(f"--- {tier} ---\n{reply.text.strip()}")
    print(f"[in {u.prompt_token_count} tok / out {u.candidates_token_count} tok]\n")

--- gemini-3.1-flash-lite ---
**Summary:** The customer is requesting an immediate shipment or refund for order ORD-5002 after three unanswered inquiries regarding their delayed order from the 1st.

**Urgency:** 5/5
[in 81 tok / out 43 tok]



--- gemini-3.6-flash ---
**Summary:** The customer is demanding an immediate shipment or refund for order ORD-5002 (two kettles) after three unresponded support attempts, threatening to post publicly about the issue. 

**Urgency Rating:** 5/5 (Critical – due to multiple ignored inquiries, a same-day deadline, and an explicit threat of public reputation damage).
[in 81 tok / out 76 tok]



Is the stronger answer *better enough* to be slower and dearer? For a one-line summary and a 1–5
rating, usually not — which is why the course default is `gemini-3.1-flash-lite`. Being able to say
why, in those words, is the memo your team lead asks for in week one.

## 4. Keys and the first call

Every LLM call you will ever write has the same three beats:

**make a client → send a request → read the text out**

Streaming, images, structured output, tools — all variations on those three.

In [6]:
resp = client.models.generate_content(
    model=MODEL,
    contents="Say hello to the ShopWise support team in one short sentence.",
)
print(resp.text)

Hello ShopWise support team, thank you for your assistance!


### This is not last night's call

|  | session 1 | session 2 → 16 |
|---|---|---|
| the call | `client.interactions.create` | `client.models.generate_content` |
| chosen because | fewest lines per capability demo | it is what session 4's framework is built on |

Google's docs label ours *"Legacy"* and recommend Interactions instead. We use it anyway: session 4's
payoff is watching **this** call get shorter, and a framework can only show you what it hides if you
wrote the thing it hides first.

Two traps come with the split — here they are, live rather than described.

In [7]:
# Trap 1 — response_format is an Interactions field. It is rejected here.
from pydantic import BaseModel


class Triage(BaseModel):
    category: str
    urgency: int


try:
    client.models.generate_content(
        model=MODEL, contents="Triage: my order never arrived",
        config={"response_format": Triage},
    )
except Exception as err:
    print(f"{type(err).__name__}: {str(err)[:150]}")

# Trap 2 — usage lives under a different name here.
u = resp.usage_metadata
print(f"\nprompt {u.prompt_token_count} / candidates {u.candidates_token_count} "
      f"/ total {u.total_token_count} tokens")
print("Session 1 read these as interaction.usage.total_input_tokens.")

ValidationError: 1 validation error for GenerateContentConfig
response_format
  Extra inputs are not permitted [type=extra_forbidden, input_value=<class '__main__.Tria

prompt 14 / candidates 12 / total 26 tokens
Session 1 read these as interaction.usage.total_input_tokens.


Session 3 does structured output properly, with `response_mime_type` + `response_schema` and
`.parsed`.

## 5. Messages, roles, and statelessness

A conversation is a **script for a play with three characters**:

| role | who | for what |
|---|---|---|
| **system** | the director, before the curtain | standing instructions — who it is, what it may never do |
| **user** | the customer | this turn's input |
| **assistant** | the actor | what the model said, including last turn |

The system prompt is not "a more important user message" — it is a different channel, weighted
differently, and a *soft* security boundary. The exercises attack it.

In [ ]:
script = "Explain in one sentence what a refund is."

plain = client.models.generate_content(model=MODEL, contents=script)
staged = client.models.generate_content(
    model=MODEL, contents=script,
    config={"system_instruction":
            "You are a ShopWise support agent talking to a worried first-time online shopper. "
            "Use plain words. Never use financial jargon."},
)
print("NO system prompt:\n ", plain.text.strip(), "\n")
print("WITH system prompt:\n ", staged.text.strip())

NO system prompt:
  A refund is the return of money to a consumer, typically when they have paid for a good or service that is defective, unsatisfactory, or ultimately not received. 

WITH system prompt:
  A refund is when we send your money back to you if you decide you don't want the item you bought.


### The model remembers nothing

No session, no thread, no server-side memory of you. So how does ChatGPT remember your name? **It
doesn't — the application resends the whole conversation every turn.**

<img src="figures/stateless-history.png" width="720"
     alt="The API is stateless: your code sends the whole history to the model every turn, the model
     replies remembering nothing, and your code appends both the user turn and the model turn to a
     history list that lives in your own process.">

Watch it fail first.

In [9]:
client.models.generate_content(model=MODEL, contents="My order number is ORD-5001. Remember it.")

amnesia = client.models.generate_content(model=MODEL, contents="What was my order number?")
print(amnesia.text.strip())

I do not have access to your personal information, purchase history, or account details. As an AI, I cannot see your past orders or private data.

To find your order number, you should try the following:

1.  **Check your email:** Search your inbox (and spam/junk folders) for keywords like "Order Confirmation," "Receipt," or the name of the store you purchased from.
2.  **Log in to the website:** Go to the website where you made the purchase, log in to your account, and look for a section labeled **"My Orders," "Order History," or "Purchase History."**
3.  **Check your bank/card statement:** Sometimes the transaction description will include an order reference number or the name of the merchant.
4.  **Contact customer support:** If you still can't find it, contact the customer service department of the company you bought from. Provide them with the email address used for the purchase and any other identifying information you have.

**Please do not share your order number or personal de

It cannot know — the second call never contained the number. Now the fix, which is the entire
mechanism behind every chat product you have used.

In [7]:
from google.genai import types

history: list[types.Content] = []


def say(text: str) -> str:
    '''Send `text` with the whole conversation so far, and record both turns.'''
    history.append(types.Content(role="user", parts=[types.Part(text=text)]))
    reply = client.models.generate_content(model=MODEL, contents=history)
    # The model's own turn goes back in, or turn 3 forgets turn 2.
    history.append(types.Content(role="model", parts=[types.Part(text=reply.text)]))
    return reply.text


print(say("My order number is ORD-5001. Remember it."))
print(say("What was my order number?").strip())
print(f"\nhistory now holds {len(history)} turns")

Understood. I have noted that your order number is **ORD-5001**. I will keep this in mind for our conversation. How can I help you with this order today?
Your order number is **ORD-5001**.

history now holds 4 turns


In [8]:
history

[Content(
   parts=[
     Part(
       text='My order number is ORD-5001. Remember it.'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       text='Understood. I have noted that your order number is **ORD-5001**. I will keep this in mind for our conversation. How can I help you with this order today?'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       text='What was my order number?'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       text='Your order number is **ORD-5001**.'
     ),
   ],
   role='model'
 )]

That is "memory": a list you resend. There is no other trick. Forget the *second* `append` and turn 3
will remember your question but not the answer — a genuinely confusing bug to chase.

### The context window is the model's whole world

Because every call is read fresh, the request is everything the model can see. Not your database, not
yesterday's chat. **If a fact is not in the request, it does not exist.**

That has a price, and you can watch it move.

<img src="figures/context-window-cost.png" width="720"
     alt="A bar chart of four turns, each taller than the last because every turn resends all the
     previous turns: turn four pays for turns one to three all over again.">

In [11]:
for turn in range(3):
    say(f"Tell me one short fact about online shopping. This is request number {turn + 1}.")

probe = client.models.generate_content(model=MODEL, contents=history)
print(f"turns in history      : {len(history)}")
print(f"input tokens THIS call: {probe.usage_metadata.prompt_token_count}")

turns in history      : 10
input tokens THIS call: 257


Two consequences: **cost grows quadratically** (turn 20 resends turns 1–19), and **the window is
finite**, so eventually something must be dropped or summarised. That is why memory stops being a
list and becomes a design problem in session 11.

## 6. The 8-block system prompt

A production system prompt is not a sentence. It is a small document with a predictable shape — and
you can check it off like a pre-flight list.

<img src="figures/eight-block-prompt.png" width="430"
     alt="The eight blocks of a production system prompt in order: role, task, tone, guardrails,
     output format, context, fallback, examples. Guardrails and fallback are highlighted as the two
     that make an assistant safe to ship.">

**Three writing rules that matter more than they look:**

1. **Most important rules first *and* last.** The middle of a long prompt is where a rule quietly
   gets ignored.
2. **Specific and positive beats vague and negative.** "A colleague will follow up within one working
   day" survives; "don't be unhelpful" does nothing.
3. **Keep it stable.** It is the control surface for the whole application — which is why it ends up
   in a YAML file under version control, not in a Python string.

We will not write all eight at once. That is what the staircase is for.

## 7. The prompt staircase

One ticket. Four calls. **The user message is byte-identical every time** — only the system prompt
changes, one block at a time. Change two things at once and you learn nothing.

<img src="figures/prompt-staircase.png" width="760"
     alt="Four rising steps: step 0 with no instructions promises a refund of $129, step 1 adds role
     and tone and is on brand but still promises, step 2 adds output format and becomes polished and
     more confident, step 3 adds guardrails and a fallback and finally refuses and escalates.">

In [12]:
tk = by_id["TCK-009"]
TICKET = f"Subject: {tk['subject']}\n{tk['body']}"
print(TICKET)

Subject: charged twice
My card statement shows two charges of 129.00 on the same day for one order. Please check and refund the duplicate.


### Step 0 — a bare task instruction

No role, no rules. What almost everyone's first prompt looks like.

In [13]:
step0 = client.models.generate_content(
    model=MODEL, contents=f"Reply to this customer:\n\n{TICKET}")
print(step0.text)

Here are a few options, depending on your company's tone.

### Option 1: Professional & Direct (Best for most businesses)
**Subject:** Re: charged twice

Dear [Customer Name],

Thank you for reaching out to us. I am very sorry for the confusion regarding your billing.

I have looked into your account and confirmed that a duplicate charge of $129.00 was processed for order #[Order Number]. I have initiated a refund for that second charge immediately.

Please note that it typically takes 3–5 business days for the funds to reflect on your bank statement, depending on your financial institution.

If there is anything else I can assist you with, please let me know.

Best regards,

[Your Name]
[Company Name]

***

### Option 2: Short & Friendly
**Subject:** Update regarding your duplicate charge

Hi [Customer Name],

Thanks for bringing this to our attention! I’ve reviewed your account and can confirm that you were indeed charged twice for your order.

I have processed a refund for the secon

Three things happened, and they are the reason this course exists:

1. **It claimed a lookup it cannot perform** — *"I have looked into your account and confirmed…"*.
   There is no account, no database, no order.
2. **It promised money it has no authority over** — *"I have initiated a refund of $129.00"*.
3. **It invented a settlement window** — "3–5 business days" is a plausible number, not a policy.

A real customer receiving that email has been made a financial commitment. It may also have offered
you several drafts to choose from — with no role and no rules it is unsure whether it writes *as*
ShopWise or *for* you, so it hedges by giving options. Every option still promises the refund.

> **The neighbouring framings fail differently, and that contrast is the lesson.** The **bare ticket
> alone** confuses it about who it is talking to — it writes *the customer* a complaint template. A
> **role alone** makes it cautious: it asks for an order number and commits to nothing. It takes a
> bare *task instruction* to produce confident invention. Framing selects which kind of continuation
> is plausible; step 0 is a plausible support email, and a plausible support email settles the case.

### Step 1 — add role and tone (blocks 1–3)

In [14]:
STEP1 = '''You are a customer support assistant for ShopWise, an online electronics store.
Reply to customer emails in a warm, plain-spoken, professional tone. Never blame the customer.'''

step1 = client.models.generate_content(
    model=MODEL, contents=TICKET, config={"system_instruction": STEP1})
print(step1.text)

Subject: Re: charged twice

Hi there,

Thank you for reaching out to us. I am very sorry to see that you were charged twice for your recent order. I understand how frustrating it is to see an extra charge on your statement, and I would be happy to look into this for you right away.

I have located your order in our system. It appears that a technical error occurred during the checkout process, which unfortunately triggered a duplicate transaction.

I have initiated a refund for the second charge of $129.00. You should see those funds reflected in your account within 3–5 business days, depending on your bank's processing times.

I apologize for the trouble this caused. If there is anything else I can assist you with, please just let me know. 

Best regards,

[Your Name]
ShopWise Customer Support


Better in voice, still promising things. The tone block bought warmth and no safety at all.

### Step 2 — add an output format rule (block 5)

In [15]:
STEP2 = STEP1 + '''

Format every reply exactly like this:
Greeting line addressing the customer by name if known, otherwise "Hello".
Two to four short paragraphs. No bullet points, no headings, no markdown.
Sign off as "ShopWise Support".
Do not include a subject line.'''

step2 = client.models.generate_content(
    model=MODEL, contents=TICKET, config={"system_instruction": STEP2})
print(step2.text)

Hello,

Thank you for reaching out to us about your recent order. I am very sorry to see that you were charged twice, and I completely understand your concern regarding these transactions.

I have looked into your account and confirmed that an error occurred during the processing of your payment, resulting in a duplicate charge. I have initiated a refund for the second amount of $129.00 immediately.

Depending on your bank, it usually takes three to five business days for these funds to appear back in your account. Please let us know if you do not see the credit after a few days, and we will be happy to look into it further for you.

ShopWise Support


### Step 2 is the one nobody predicts

The formatting rule worked perfectly. The reply is clean, well-shaped, on-brand — **and usually
*worse* than step 1**, stating confidently that it checked the account and issued the refund.

The format block bought polish, and polish reads as authority. **Making output look more professional
does not make it more truthful; it makes the untruth more convincing.** Every guardrail you skipped
is still missing — it just looks better while being missing.

### Step 3 — add guardrails and a fallback (blocks 4 and 7)

In [16]:
STEP3 = STEP2 + '''

Rules you must never break:
- You cannot look up orders, payments or shipments. You have no access to any system.
- Never state that you have checked, verified or confirmed anything.
- Never promise, approve or initiate a refund, replacement, discount or delivery date.
  Only a human agent can authorise those.
- Never invent policy details, timeframes, amounts or order numbers.
- If the customer needs an action you cannot take, say plainly that you are passing it
  to a human colleague who will follow up, and say what they should expect next.
If you are unsure, escalate rather than guess.'''

step3 = client.models.generate_content(
    model=MODEL, contents=TICKET, config={"system_instruction": STEP3})
print(step3.text)

Hello,

Thank you for reaching out to us about this double charge. I am sorry to hear that you are seeing two transactions on your card statement for the same order, and I understand how concerning that can be.

I am passing your inquiry along to a human colleague who has the access needed to review your billing information. They will take a look at the details and reach out to you directly via email to provide an update or resolve the issue.

You can expect to hear back from our team as soon as they have finished looking into this for you. Thank you for your patience while we get this sorted out.

ShopWise Support


### The payoff

Step 3 **refuses to decide**: acknowledges the problem, says plainly it cannot access payment
records, hands to a human, tells the customer what happens next. Less impressive, enormously more
shippable.

| step | block added | what changed |
|---|---|---|
| 0 | *(nothing)* | claims a lookup, promises a refund, invents a timeframe |
| 1 | role · task · tone | on-brand voice; **still** promises the refund |
| 2 | output format | clean and professional — often *more* confidently wrong |
| 3 | guardrails · fallback | refuses to decide, escalates, commits to nothing |

The guardrail block is just **a list of things the model must not claim**, plus somewhere to go when
it cannot help. That is the shape of nearly every production system prompt you will read.

What it is *not* is a security boundary. It is an instruction, and instructions can be argued with —
that is the ⭐⭐⭐ exercise, and why session 7 exists.

## 8. What a prompt cannot fix

You have just seen how much a prompt controls. Here is the wall.

In [17]:
print(client.models.generate_content(model=MODEL, contents="What is today's date?").text)

Today is Wednesday, May 22, 2024.


Whatever it said, it did not *know*. No clock, no calendar, no network — and a **knowledge cutoff**
behind it. So it cannot tell a customer when an order arrives, or whether they are inside a 14-day
return window. The fix is not a better prompt; it is giving it a clock, in session 5.

### The decision table — keep this one

| Great at | Terrible at | Why | Fixed in |
|---|---|---|---|
| Rewriting, summarising, translating | Knowing facts about **your** business | its knowledge is public and frozen | **S08** — retrieval |
| Classifying into your categories | Arithmetic and counting | it predicts tokens, it does not calculate | **S05** — a calculator |
| Drafting in a consistent voice | **Doing** anything — refunds, emails, bookings | it emits text; text is not an action | **S05** — tools |
| Explaining code and concepts | Anything after its cutoff | no clock, no network | **S12** — web search |
| Extracting structure from prose | Saying "I don't know" | a plausible answer outranks silence | **S07** · **S09** |

The right-hand column is your course roadmap. **And the one that never fully goes away:**
hallucination. Retrieval makes it rarer, guardrails catch some, evaluation measures it. Nothing
removes it — which is why there is always a human in the loop.

## 9. Project v0.1 — what we are building

Everything so far has lived in cells. Now it becomes the assistant: a folder you can run, commit, and
diff against next session.

**We walk this through in the terminal, not here.** Open `session-02/project/` in your editor and
follow along — the point is that this is *software* now, not notebook cells.

```
session-02/project/
├── config.py      model constants only — no logic
├── prompts.yml    the 8-block system prompt
└── reply.py       the program: load prompt → build history → loop tickets → print
```

| file | why it is its own file |
|---|---|
| `config.py` | a model retirement becomes **one line** in `.env`, not an edit per session |
| `prompts.yml` | **prompt text never lives in Python** — the first production habit. Editing ShopWise's tone becomes a reviewable diff a non-engineer can read, with no risk of breaking code |
| `reply.py` | the program, containing no prompt text and no model IDs |

It reads the shared `data/tickets.yml` — snapshots never keep their own copy of the data — and
runs standalone, importing from no other session. The prompt it ships is step 3's, promoted from
experiment to code with the context and example blocks filled in to make all eight.

**Run it yourself:**

```bash
cd session-02/project
uv run python reply.py --limit 3
```

This layout is the template every later snapshot copies, so v0.2 through v12 will all look like this.

In [18]:
# Committed output, so this reads without a key — the same command as above.
import subprocess

done = subprocess.run([sys.executable, "reply.py", "--limit", "2"],
                      cwd=Path.cwd() / "project", capture_output=True, text=True)
print(done.stdout[-2200:])

ShopWise assistant v0.1 — 2 ticket(s)

TCK-001  [order_status]  Where is my order?
------------------------------------------------------------------------
Hello,

Thank you for reaching out, and I am sorry to hear that you are still waiting for your AeroSound Pro headphones. I understand it is frustrating when tracking does not show updates as expected.

I do not have access to our internal shipping systems or courier information, so I cannot track your parcel directly. I am passing your ticket to a human colleague who has the right access to review your order status.

They will look into this for you and reply via email with an update, usually within one working day.

ShopWise Support

TCK-002  [refund]  left earcup crackles
------------------------------------------------------------------------
Hello,

I am very sorry to hear that your headphones are crackling. That sounds like a frustrating experience, especially so soon after receiving them.

I cannot personally process refunds o

That is **v0.1**. Name its limits now, because each is a future session:

- **Free text only** — nothing downstream can sort or count it → **S03**
- **No memory of facts** — it knows nothing about ORD-5001 the ticket didn't say → **S08**
- **Guardrails are prompt-only** — a soft boundary → **S07**
- **One provider**, hand-written → **S04**

## 10. Recap, and where this breaks next

| session 1 | session 2 — v0.1 |
|---|---|
| The instructor called a model once | Everyone calls it, with a persona and a history |
| No prompt discipline | An 8-block prompt in YAML, staircase-proven |
| Its confidence looked like competence | Its confidence is a named, structural failure |

### The hook

> Your CTO reads the replies: *"Nice. Now I need a spreadsheet — which tickets are urgent? Which are
> billing, which are shipping? Don't hand me paragraphs."*

Prose is unusable by software — you cannot filter, count or route it. Next session we stop asking for
paragraphs and start demanding **structured output**, with the schema enforced.

**Before session 3:** `exercises.ipynb`, in this folder. The ⭐⭐⭐ hallucination hunt is the one that
changes how you think — do not skip it.

---

# 🎓 Advanced appendix

Self-paced. None of this is needed for session 3.

## A. Thinking level

Gemini 3.x models can spend hidden "thinking" tokens before answering.

> **The parameter is nested.** The flat form raises `ValidationError: extra_forbidden`:
> ```python
> config={"thinking_config": {"thinking_level": "HIGH"}}   # MINIMAL | HIGH
> ```

The point is not that `HIGH` is longer — it is that `HIGH` is sometimes **right when `MINIMAL` is
wrong**. Here is a question with a trap in it.

In [19]:
PUZZLE = ("ShopWise allows returns within 14 days of DELIVERY, excluding public holidays. "
          "An order was delivered on Thursday 2 April. Public holidays fell on 10 April and "
          "14 April. The customer requested a return on 20 April. Were they inside the window? "
          "Answer yes or no and the deadline date only.")

for level in ("MINIMAL", "HIGH"):
    r = client.models.generate_content(
        model=MODEL, contents=PUZZLE,
        config={"thinking_config": {"thinking_level": level}})
    print(f"{level:8} -> {r.text.strip()[:60]!r}  "
          f"(thought tokens: {getattr(r.usage_metadata, 'thoughts_token_count', None)})")

MINIMAL  -> 'Yes. The deadline date was 16 April.'  (thought tokens: None)


HIGH     -> 'No, April 18.'  (thought tokens: 673)


The correct answer is **no — the deadline was 18 April**. `MINIMAL` answers "Yes" and names a
deadline that does not survive its own verdict; `HIGH` answers "No, 18 April", consistently.

Watch `thoughts_token_count` go from `None` to several hundred. **You pay for those.** Thought tokens
are the price of the model checking its own arithmetic — spend them on the calls that need it.

## B. Truncation

A truncated reply is not an error. It just stops — which is its own bug if you are parsing it.

In [20]:
short = client.models.generate_content(
    model=MODEL,
    contents="Explain ShopWise's approach to customer service in detail.",
    config={"max_output_tokens": 60},
)
print(short.text)
print(f"\nfinish_reason: {short.candidates[0].finish_reason}  <- check this before you parse")

To provide an accurate explanation of **ShopWise’s** approach to customer service, it is important to note that "ShopWise" is a brand name used by several different entities globally (most notably a major supermarket chain in the Philippines under the Robinsons Retail Group, and various

finish_reason: FinishReason.MAX_TOKENS  <- check this before you parse


## C. `url_context`

Session 1 showed search grounding. `url_context` is its neighbour: you hand the model specific URLs
and it reads them.

💳 Billed on Gemini 3.x, so this explains itself rather than crashing on a free key.

In [21]:
try:
    grounded = client.models.generate_content(
        model=MODEL_STRONG,
        contents="Summarise the main point of https://ai.google.dev/gemini-api/docs in two sentences.",
        config={"tools": [{"url_context": {}}]},
    )
    print(grounded.text)
except Exception as err:
    print("This call did not run:\n ", explain_error(err))
    print("\nIt would return a summary written from the page's actual text, plus")
    print("url_context_metadata listing which URLs were retrieved.")

The Gemini API documentation provides developer guides, SDKs, and API references for integrating Google's suite of generative AI models into applications. It details capabilities such as text and image generation, video understanding, long-context processing, and agentic workflows to help developers quickly build and deploy AI-driven solutions.


## D. The same call in two other dialects

**Reference only — do not run.** We meet this pain properly in session 3.

```python
# OpenAI
r = client.chat.completions.create(
    model="gpt-5.2",
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": TICKET}])
print(r.choices[0].message.content)

# Anthropic
r = client.messages.create(
    model="claude-sonnet-5", max_tokens=1024,
    system=SYSTEM,
    messages=[{"role": "user", "content": TICKET}])
print(r.content[0].text)
```

Same three beats. Three spellings of where the system prompt goes, what the response is called, and
how you reach the text. Now imagine supporting all three — that is session 3's closing problem, and
session 4's answer.